In [18]:
#import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer


In [19]:
#load dataset
df=pd.read_csv('/content/titanic.csv')
df


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [20]:
#check null values

print(df.info())
print(df.isnull().sum())
#print(df.describe(include='all'))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
None
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int6

In [21]:
df.isnull().sum()



,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [22]:
#deop columns
df = df.drop(columns=['Cabin'], errors='ignore')
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S
...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C


In [23]:
#fill column with median
df['Age'] = df['Age'].fillna(df['Age'].median())

In [24]:
##fill column with mode
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

In [36]:
numerical_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print("Numerical features identified:", numerical_features)

Numerical features identified: ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']


In [37]:
for col in numerical_features:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    print(f"Column: {col}")
    print(f"  Q1: {Q1}")
    print(f"  Q3: {Q3}")
    print(f"  IQR: {IQR}\n")

Column: Survived
  Q1: 0.0
  Q3: 1.0
  IQR: 1.0

Column: Pclass
  Q1: 2.0
  Q3: 3.0
  IQR: 1.0

Column: Age
  Q1: 22.0
  Q3: 35.0
  IQR: 13.0

Column: SibSp
  Q1: 0.0
  Q3: 1.0
  IQR: 1.0

Column: Parch
  Q1: 0.0
  Q3: 0.0
  IQR: 0.0

Column: Fare
  Q1: 7.9104
  Q3: 31.0
  IQR: 23.0896



In [47]:

for col in numerical_features:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outlier_bounds[col] = {'lower_bound': lower_bound, 'upper_bound': upper_bound}
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"Number of outliers in column {col}: {len(outliers)}")
    print(f"Column: {col}")
    print(f"  Lower Bound: {lower_bound}")
    print(f"  Upper Bound: {upper_bound}\n")

Number of outliers in column Survived: 0
Column: Survived
  Lower Bound: -1.5
  Upper Bound: 2.5

Number of outliers in column Pclass: 0
Column: Pclass
  Lower Bound: 0.5
  Upper Bound: 4.5

Number of outliers in column Age: 66
Column: Age
  Lower Bound: 2.5
  Upper Bound: 54.5

Number of outliers in column SibSp: 46
Column: SibSp
  Lower Bound: -1.5
  Upper Bound: 2.5

Number of outliers in column Parch: 213
Column: Parch
  Lower Bound: 0.0
  Upper Bound: 0.0

Number of outliers in column Fare: 116
Column: Fare
  Lower Bound: -26.724
  Upper Bound: 65.6344



In [48]:
X = df.drop('Survived', axis=1)
y = df['Survived']

In [49]:
scaler = StandardScaler()
numeric_cols = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']

X[numeric_cols] = scaler.fit_transform(X[numeric_cols])

In [50]:
print(X.head())
print(X.shape)

     Pclass       Age     SibSp     Parch      Fare  Sex_female  Sex_male  \
0  0.827377 -0.565736  0.432793 -0.473674 -0.502445       False      True   
1 -1.566107  0.663861  0.432793 -0.473674  0.786845        True     False   
2  0.827377 -0.258337 -0.474545 -0.473674 -0.488854        True     False   
3 -1.566107  0.433312  0.432793 -0.473674  0.420730        True     False   
4  0.827377  0.433312 -0.474545 -0.473674 -0.486337       False      True   

   Embarked_C  Embarked_Q  Embarked_S  
0       False       False        True  
1        True       False       False  
2       False       False        True  
3       False       False        True  
4       False       False        True  
(891, 10)
